# 06a — Golden Eval Set & RAGAS
**Domain:** Science / arXiv RAG (Module 3 pipeline) &nbsp;|&nbsp; **Time:** ~2 h  
**Key Concepts:** golden Q&A set, RAGAS metrics, failure analysis

---
### RAGAS Metrics
| Metric | Measures |
|--------|----------|
| **faithfulness** | Is the answer grounded in the retrieved context? |
| **answer_relevancy** | Does the answer actually address the question? |
| **context_precision** | Are retrieved chunks relevant? |
| **context_recall** | Did retrieval find the right chunks? |


In [ ]:
import sys, os
sys.path.insert(0, '.')
from llm_checker import check, hint, evaluate, progress, show_exercise
print("✅ ready")


In [ ]:
try:
    from ragas import evaluate as ragas_eval
    from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
    print("✅ RAGAS available")
except ImportError:
    print("❌ pip install ragas")
    ragas_eval = None


---
## 🔵 Example — Ex 06a-1: RAGAS Eval on a Mini Golden Set

In [ ]:
from openai import OpenAI
from datasets import Dataset

client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

def mock_rag(question: str) -> dict:
    """Minimal mock RAG for demonstration."""
    context = ("Transformer models use self-attention to process all tokens in parallel, "
               "unlike RNNs which process sequentially.")
    resp = client.chat.completions.create(
        model="local-model",
        messages=[{"role": "user",
                   "content": f"Context: {context}\nQuestion: {question}\nAnswer briefly:"}],
        max_tokens=100,
    )
    return {"answer": resp.choices[0].message.content.strip(), "contexts": [context]}

GOLDEN_5 = [
    ("What is self-attention?",    "Self-attention lets each token attend to all other tokens in the sequence."),
    ("How do transformers differ from RNNs?", "Transformers process all tokens in parallel; RNNs process sequentially."),
    ("What is BERT?",              "BERT is a bidirectional transformer pre-trained on masked language modelling."),
    ("How does GPT work?",         "GPT uses a decoder-only transformer to predict the next token autoregressively."),
    ("What are embeddings?",       "Embeddings are dense vector representations of tokens or sentences."),
]

eval_data = []
for question, ground_truth in GOLDEN_5:
    out = mock_rag(question)
    eval_data.append({"question": question, "answer": out["answer"],
                      "contexts": out["contexts"], "ground_truth": ground_truth})

ds_demo = Dataset.from_list(eval_data)
print(f"Demo dataset: {ds_demo}")


In [ ]:
if ragas_eval:
    try:
        demo_result = ragas_eval(ds_demo, metrics=[faithfulness, answer_relevancy,
                                                    context_precision, context_recall])
        print("RAGAS demo result:", demo_result)
    except Exception as e:
        print(f"RAGAS call failed: {e}  (needs OpenAI key or configured LLM judge)")
else:
    print("RAGAS not installed — skipping demo eval")


---
## 🟡 Exercise — Ex 06a-2: 50-pair Golden Eval Set

In [ ]:
show_exercise(
    "06a-2", "Build 50-pair golden eval set + RAGAS evaluation",
    """1. Load arXiv abstracts (or use mock_rag fallback).
2. For each of 50 abstracts: generate one question + ground_truth via LLM.
3. Run your RAG pipeline to get answer + contexts.
4. Save to golden_eval_arxiv.jsonl.
5. Run RAGAS on all 50 pairs and report all 4 metrics.
6. Identify the top-5 failure cases by lowest faithfulness score.""",
    hints=[
        "Generate Q+GT: LLM prompt → JSON {question, ground_truth} from the abstract",
        "save: json.dumps(item) + '\\n' per line",
        "Failures: df.nsmallest(5, 'faithfulness')[['question','faithfulness']]",
    ],
    checks=[
        "golden_eval_arxiv.jsonl exists with >= 50 lines",
        "RAGAS eval returns all 4 metrics (or skipped with note if no API key)",
        "Top-5 failures identified",
    ],
)


In [ ]:
import json, pandas as pd

DATA_PATH = os.path.join("data", "arxiv", "arxiv_abstracts_5k.csv")
if os.path.exists(DATA_PATH):
    arxiv_df = pd.read_csv(DATA_PATH).head(50)
    print(f"✅ Loaded {len(arxiv_df)} abstracts")
else:
    print("⚠️  arXiv data missing — using synthetic fallback")
    arxiv_df = pd.DataFrame({"abstract": [
        f"We present a novel method for {topic} using deep neural networks."
        for topic in ["image classification","text generation","speech recognition",
                      "machine translation","object detection"] * 10
    ]})


def generate_qa_from_abstract(abstract: str) -> dict:
    """LLM generates a question + concise ground-truth from an abstract."""
    resp = client.chat.completions.create(
        model="local-model",
        messages=[{"role": "user", "content":
            f"Read this abstract and write ONE specific question and a concise ground-truth answer.\n"
            f"Abstract: {abstract[:500]}\n"
            "Reply ONLY as valid JSON: {\"question\": \"...\", \"ground_truth\": \"...\"}"}],
        max_tokens=150,
    )
    raw = resp.choices[0].message.content.strip()
    try:
        return json.loads(raw[raw.find("{"):])
    except Exception:
        return {"question": "What is this paper about?", "ground_truth": abstract[:80]}


golden_pairs = []
eval_data_50  = []
for _, row in arxiv_df.iterrows():
    abstract = str(row.get("abstract", row.iloc[0]))[:600]
    qa  = generate_qa_from_abstract(abstract)
    out = mock_rag(qa["question"])
    item = {"question": qa["question"], "ground_truth": qa["ground_truth"],
            "answer": out["answer"], "contexts": out["contexts"]}
    golden_pairs.append(item)
    eval_data_50.append(item)

with open("golden_eval_arxiv.jsonl", "w") as f:
    for p in golden_pairs:
        f.write(json.dumps(p) + "\n")
print(f"✅ Saved {len(golden_pairs)} pairs to golden_eval_arxiv.jsonl")


In [ ]:
ragas_result_df = None
top5_failures  = None

if ragas_eval and eval_data_50:
    try:
        ds50 = Dataset.from_list(eval_data_50)
        ragas_result = ragas_eval(ds50, metrics=[faithfulness, answer_relevancy,
                                                  context_precision, context_recall])
        ragas_result_df = ragas_result.to_pandas()
        top5_failures = ragas_result_df.nsmallest(5, "faithfulness")[["question", "faithfulness"]]
        print("RAGAS results:\n", ragas_result_df[["faithfulness","answer_relevancy"]].describe())
        print("\nTop-5 failures:\n", top5_failures)
    except Exception as e:
        print(f"⚠️  RAGAS eval skipped: {e}  (set OPENAI_API_KEY to run full eval)")
else:
    print("ℹ️  RAGAS skipped (not installed or no data)")

check([
    (os.path.exists("golden_eval_arxiv.jsonl"),   "golden_eval_arxiv.jsonl exists"),
    (len(golden_pairs) >= 50,                     f">= 50 pairs (got {len(golden_pairs)})"),
    (ragas_result_df is not None or True,         "RAGAS eval run (or noted as skipped)"),
], exercise_id="06a-2")


In [ ]:
check([
    (os.path.exists("golden_eval_arxiv.jsonl"), "Golden eval set saved"),
    (len(golden_pairs) >= 50,                   "50 Q&A pairs created"),
], exercise_id="06a-final")
progress()
